In [17]:
from constants import users_list, data_path
from lib import spoti, genre_normalizer, plotting, preprocessing, dimensionality_reduction

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import json
import os
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import mean_squared_error
import random
import time
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors, NeighborhoodComponentsAnalysis
from sklearn.metrics import pairwise_distances
import pickle
import numpy as np
from typing import List, Dict, Tuple, Optional, Any, Callable
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from IPython.display import display, HTML

# Import data

In [18]:
df = spoti.load_all_tracks(
    base_path=data_path.DATA_PATH,
    users=users_list.USERS,
    load_spotify_tracks=False,
    penality_factors={"short_term": 1, "medium_term": 0.5, "long_term": 0.1},
)
df

,album,artists,available_markets,disc_number,duration_ms,explicit,external_ids,external_urls,href,id,...,time_range,affinity,username,release_year,normalized_genres,added_at,episode,track,added_by,playlist_id
0,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,196426,False,{'isrc': 'USSM12301260'},{'spotify': 'https://open.spotify.com/track/75...,https://api.spotify.com/v1/tracks/75rqqKvzJCGv...,75rqqKvzJCGv2oq9C4yFDt,...,medium_term,0.50,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
1,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,137533,True,{'isrc': 'USSM12109218'},{'spotify': 'https://open.spotify.com/track/2F...,https://api.spotify.com/v1/tracks/2FYGZDfsAnNs...,2FYGZDfsAnNsrm1gVbyKnG,...,medium_term,0.49,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
2,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,162906,True,{'isrc': 'USSM12109222'},{'spotify': 'https://open.spotify.com/track/4k...,https://api.spotify.com/v1/tracks/4kroNlz8BTfs...,4kroNlz8BTfswE4M0i3YCh,...,medium_term,0.48,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
3,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,89749,False,{'isrc': 'USSM12208854'},{'spotify': 'https://open.spotify.com/track/2N...,https://api.spotify.com/v1/tracks/2N3YZ075lq9z...,2N3YZ075lq9z1ObaAiX6l1,...,medium_term,0.47,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
4,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,174044,False,{'isrc': 'USSM12300114'},{'spotify': 'https://open.spotify.com/track/2S...,https://api.spotify.com/v1/tracks/2SiAcexM2p1y...,2SiAcexM2p1yX6joESbehd,...,medium_term,0.46,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12424,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AL, AM, AT, AZ, BA, BE, BG, BY, CH, CW, C...",1,251880,False,{'isrc': 'GBN9Y1100001'},{'spotify': 'https://open.spotify.com/track/3z...,https://api.spotify.com/v1/tracks/3z7dWKRsjDNM...,3z7dWKRsjDNM24ohLKZBnA,...,NaN,NaN,dany,1967,"[rock, rock, rock, rock, rock, rock, rock]",2022-12-30 08:42:31+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12425,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,193853,False,{'isrc': 'GBLTP1700005'},{'spotify': 'https://open.spotify.com/track/1V...,https://api.spotify.com/v1/tracks/1VofMhhL98pe...,1VofMhhL98pewltVGBSmCW,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:07+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12426,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,295493,False,{'isrc': 'GBLTP1700011'},{'spotify': 'https://open.spotify.com/track/0K...,https://api.spotify.com/v1/tracks/0KE7apgczHNY...,0KE7apgczHNYiXIvMUY0Fc,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:13+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12427,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,467306,False,{'isrc': 'GBLTP1700014'},{'spotify': 'https://open.spotify.com/track/6v...,https://api.spotify.com/v1/tracks/6vbRA9yAAgIX...,6vbRA9yAAgIXtDlmhyNqPq,...,NaN,N

# Logistic Matrix Factorization for Implicit Feedback Data (Logistic MF)
From [Christopher C. Johnson - Logistic Matrix Factorization for Implicit Feedback Data](https://web.stanford.edu/~rezab/nips2014workshop/submits/logmat.pdf)

## Setup the dataset

In [19]:
df_matrix_mf = df.copy()
df_matrix_mf = df_matrix_mf[df["type"] == "top_track"]
df_matrix_mf["username"] = df_matrix_mf["username"].astype("category")
df_matrix_mf["id"] = df_matrix_mf["id"].astype("category")
df_matrix_mf[["username", "id", "affinity"] + spoti.NUMERICAL_FEATURES]
df_matrix_mf["affinity"] *= 100
df_matrix_mf["affinity"]

0        50.0
1        49.0
2        48.0
3        47.0
4        46.0
         ... 
12318     1.0
12319     0.8
12320     0.6
12321     0.4
12322     0.2
Name: affinity, Length: 1200, dtype: float64

In [20]:
# Create the matrix dataset
class MatrixDataset:
    _matrix: np.ndarray

    @property
    def matrix(self) -> np.ndarray:
        return self._matrix

    def __init__(self, num_users: int, num_items: int):
        # A matrix of shape (num_users, num_items)
        self._matrix = np.zeros((num_users, num_items))

    def add_interaction(self, user_id: int, item_id: int, value: float):
        """
        Adds a new interaction to the matrix.
        :param user_id: The user id.
        :param item_id: The item id.
        :param value: The value of the interaction.
        """
        self._matrix[user_id, item_id] = value

    def fill_from_df(self, users: pd.Series, items: pd.Series, values: pd.Series):
        """
        Fills the matrix from a dataframe.
        :param df: The dataframe.
        :param user_col: The user column.
        :param item_col: The item column.
        :param value_col: The value column.
        """
        assert (
            len(users) == len(items) == len(values)
        ), "The length of the users, items and values must be the same."

        for user, item, value in zip(users, items, values):
            self.add_interaction(user, item, value)

    def __str__(self):
        return str(self._matrix)

In [21]:
num_users = len(df_matrix_mf["username"].unique())
num_items = len(df_matrix_mf["id"].unique())

In [22]:
matrix_mf = MatrixDataset(num_users, num_items)
matrix_mf.fill_from_df(df_matrix_mf["username"].cat.codes, df_matrix_mf["id"].cat.codes, df_matrix_mf["affinity"])
R = matrix_mf.matrix
R

array([[ 0. ,  0. , 62. , ...,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. , ...,  0. ,  0. , 54. ],
       ...,
       [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
       [ 4.4, 30. ,  0. , ...,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. , ..., 46. ,  0. ,  0. ]])

In [23]:
def convert_to_ids(values: List[str], column: str) -> List[int]:
    """
    Gets the user ids from the usernames.
    :param usernames: The usernames.
    :return: The user ids.
    """
    return df_matrix_mf[df_matrix_mf[column].isin(values)][column].cat.codes.tolist()

def retrieve_value_from_ids(ids: List[int], column: str) -> str:
    """
    Gets the value from the ids.
    :param ids: The ids.
    :return: The value.
    """
    return df_matrix_mf[df_matrix_mf[column].cat.codes.isin(ids)][column].tolist()

def get_df_rows_from_ids(ids: List[int], column: str, search_in: pd.DataFrame) -> pd.DataFrame:
    """
    Gets the dataframe rows from the ids.
    :param ids: The ids.
    :return: The dataframe rows.
    """
    return search_in[search_in[column].cat.codes.isin(ids)]

In [24]:
# Create a matrix U that contains the index that sorts the users by their affinity
I = np.argsort(matrix_mf.matrix, axis=1)
I.shape

(8, 923)

In [25]:
def compute_alpha(matrix: np.ndarray) -> np.ndarray:
    """
    Computes the alpha matrix.
    :param matrix: The matrix.
    :param k: The number of neighbors.
    :return: The alpha matrix.
    """
    num_items = matrix.shape[1] * matrix.shape[0]
    sum_matrix = matrix.flatten().sum()
    number_of_zeros = num_items - np.count_nonzero(matrix)
    alpha = number_of_zeros / sum_matrix
    return alpha

alpha = compute_alpha(matrix_mf.matrix)
print(alpha)
R *= alpha
R

0.264317361339022


array([[ 0.        ,  0.        , 16.3876764 , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        , 14.27313751],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 1.16299639,  7.92952084,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ..., 12.15859862,
         0.        ,  0.        ]])

_Let $l_{u,i}$ denote the event that user $u$ has chosen to interact with item $i$ (user $u$ prefers item $i$). Then, we can let the probability of this event occurring be distributed according to a logistic function parameterized by the sum of the inner product of user and item latent factor vectors and user and item biases._
$$
p(l_{ui} | x_u, y_i, \beta_i, \beta_j) = \frac{\exp(x_uy_i^T + \beta_u + \beta_i)}{1 + \exp(x_uy_i^T + \beta_u + \beta_i)}
$$



The log posterior probability of $x_u, y_i, \beta_u, \beta_i$ given the observed data $\mathbf{R}$ is given by:
$$
\log p(\mathbf{X}, \mathbf{Y}, \beta | \mathbf{R}) = \sum_{u,i} \alpha r_{ui} (x_u^T y_i + \beta_u + \beta_i) - (1 + \alpha r_{ui}) \log(1 + \exp(x_u^T y_i + \beta_u + \beta_i)) - \frac{\lambda}{2} ||x_u||^2 - \frac{\lambda}{2} ||y_i||^2
$$

In [26]:
def log_posterior(
    X: torch.Tensor,
    Y: torch.Tensor,
    beta_u: torch.Tensor,
    beta_i: torch.Tensor,
    R: torch.Tensor,
    alpha: float,
    lambd: float,
) -> torch.Tensor:
    """
    Computes the log posterior of the model, which is:
    :param X: The latent vectors of the users.
    :param Y: The latent vectors of the items.
    :param beta_u: The bias of the users.
    :param beta_i: The bias of the items.
    :param R: The matrix of interactions.
    :param alpha: The alpha parameter.
    :param lambd: The lambda parameter.
    :return: The log posterior.
    """
    # Compute the first term
    term1 = alpha * R * (torch.matmul(X, Y.t()) + beta_u[:, None] + beta_i)
    term2 = (1 + alpha * R) * torch.log1p(torch.exp(torch.matmul(X, Y.t()) + beta_u[:, None] + beta_i))
    regularization = (lambd / 2) * (torch.norm(X, p=2) ** 2 + torch.norm(Y, p=2) ** 2)
    return torch.sum(term1 - term2) - regularization

In [27]:
def mpr(I: torch.Tensor, R: torch.Tensor) -> float:
    """
    Compute the Mean Percentile Ranking (MPR) using a sorted index matrix.

    :param I: Matrix of sorted indices of items for each user.
    :param R: Rating or interaction matrix.
    :return: MPR value.
    """
    num_users, num_items = R.shape
    total_interactions = torch.sum(R)

    # Initialize MPR
    mpr = 0.0

    # Iterate over each user
    for u in range(num_users):
        # Get the indices of the items in sorted order for this user
        sorted_indices = I[u]

        # Calculate the rank for each item
        for i in range(num_items):
            item_index = sorted_indices[i]
            rank = i / num_items  # Percentile rank
            mpr += R[u, item_index] * rank

    # Normalize by the total number of interactions
    mpr /= total_interactions

    return mpr

In [28]:
def compute_predictions(
    X: torch.Tensor,
    Y: torch.Tensor,
    beta_u: torch.Tensor,
    beta_i: torch.Tensor,
) -> torch.Tensor:
    """
    Computes the predictions of the model.
    :param X: The latent vectors of the users.
    :param Y: The latent vectors of the items.
    :param beta_u: The bias of the users.
    :param beta_i: The bias of the items.
    :param R: The matrix of interactions.
    :return: The predictions.
    """
    return torch.matmul(X, Y.t()) + beta_u[:, None] + beta_i

In [29]:
R[np.isnan(R)] = 0

In [30]:
def train(
    R: np.ndarray,
    num_users: int,
    num_items: int,
    num_latent: int,
    lambd: float,
    alpha: float,
    learning_rate: float,
    epochs: int,
):
    """
    Trains the model.
    :param num_latent: The number of latent factors.
    :param lambd: The lambda parameter.
    :param alpha: The alpha parameter.
    :param learning_rate: The learning rate.
    :param epochs: The number of epochs.
    """
    mprs = []  # Record MPRs for each epoch
    losses = []  # Record losses for each epoch

    # Initialize user and item latent factor matrices and bias vectors
    R = torch.tensor(R, dtype=torch.float64)
    X = torch.randn(num_users, num_latent, requires_grad=False, dtype=torch.float64)
    Y = torch.randn(num_items, num_latent, requires_grad=False, dtype=torch.float64)
    beta_u = torch.randn(num_users, requires_grad=False, dtype=torch.float64)
    beta_i = torch.randn(num_items, requires_grad=False, dtype=torch.float64)

    # Initialize optimizer
    optimizer = optim.Adagrad([X, Y, beta_u, beta_i], lr=learning_rate)

    # Training loop
    for epoch in range(epochs):
        # Fix X and B and take a step toward Y and B
        X.requires_grad = False
        Y.requires_grad = True
        beta_u.requires_grad = False
        beta_i.requires_grad = True
        optimizer.zero_grad()
        loss = -log_posterior(X, Y, beta_u, beta_i, R, alpha, lambd)
        loss.backward()
        optimizer.step()

        # Fix Y and B and take a step toward X and B
        X.requires_grad = True
        Y.requires_grad = False
        beta_u.requires_grad = True
        beta_i.requires_grad = False
        optimizer.zero_grad()
        loss = -log_posterior(X, Y, beta_u, beta_i, R, alpha, lambd)
        loss.backward()
        optimizer.step()

        # Make predictions
        predictions = compute_predictions(X, Y, beta_u, beta_i)

        # Compute MPR
        I = torch.argsort(predictions, descending=True, dim=1)
        mpr_value = mpr(I, R)

        # Record MPR
        mprs.append(mpr_value)

        # Record loss
        losses.append(loss.item())

        # Print progress
        print(f"Epoch {epoch + 1} - Loss: {loss.item():.4f} - MPR: {mpr_value:.4f}")

    return X, Y, beta_u, beta_i, mprs, losses

In [31]:
def train_with_gradients(
    R: np.ndarray,
    num_users: int,
    num_items: int,
    num_latent: int,
    lambd: float,
    alpha: float,
    learning_rate: float,
    epochs: int,
):
    """
    Trains the model.
    :param num_latent: The number of latent factors.
    :param lambd: The lambda parameter.
    :param alpha: The alpha parameter.
    :param learning_rate: The learning rate.
    :param epochs: The number of epochs.
    """
    mprs = []  # Record MPRs for each epoch
    losses = []  # Record losses for each epoch

    # Initialize user and item latent factor matrices and bias vectors
    R = torch.tensor(R, dtype=torch.float64)
    X = torch.randn(num_users, num_latent, requires_grad=False, dtype=torch.float64)
    Y = torch.randn(num_items, num_latent, requires_grad=False, dtype=torch.float64)
    beta_u = torch.randn(num_users, requires_grad=False, dtype=torch.float64)
    beta_i = torch.randn(num_items, requires_grad=False, dtype=torch.float64)

    grad_accumulator_X = torch.zeros_like(X)
    grad_accumulator_Y = torch.zeros_like(Y)

    for epoch in range(epochs):
        # Fix X and B and take a step toward Y and B
        for i in range(num_items):
            term1 = alpha * R[:, i][:, None]

            exp_term = torch.exp(torch.matmul(Y[i], X.t()) + beta_u + beta_i[i])
            exp_term = exp_term[:, None]

            gradients_Y = torch.sum(term1 * X - X * (1 + term1) * exp_term / (1 + exp_term), dim=0) - lambd * Y[i]
            gradients_beta_i = torch.sum(term1 - (1 + term1) * exp_term / (1 + exp_term), dim=0).squeeze()

            grad_accumulator_Y[i] += gradients_Y ** 2
            Y[i] += learning_rate * gradients_Y / torch.sqrt(grad_accumulator_Y[i])
            beta_i[i] += learning_rate * gradients_beta_i
        
        # Fix Y and B and take a step toward X and B
        for u in range(num_users):
            term1 = alpha * R[u, :][:, None]

            exp_term = torch.exp(torch.matmul(X[u], Y.t()) + beta_u[u] + beta_i)
            exp_term = exp_term[:, None]

            gradients_X = torch.sum(term1 * Y - Y * (1 + term1) * exp_term / (1 + exp_term), dim=0) - lambd * X[u]
            gradients_beta_u = torch.sum(term1 - (1 + term1) * exp_term / (1 + exp_term), dim=0).squeeze()

            grad_accumulator_X[u] += gradients_X ** 2
            X[u] += learning_rate * gradients_X / torch.sqrt(grad_accumulator_X[u])
            beta_u[u] += learning_rate * gradients_beta_u

        # Make predictions
        predictions = compute_predictions(X, Y, beta_u, beta_i)

        # Compute MPR
        I = torch.argsort(predictions, descending=True, dim=1)
        mpr_value = mpr(I, R)

        # Record MPR
        mprs.append(mpr_value)

        # Record loss
        loss = log_posterior(X, Y, beta_u, beta_i, R, alpha, lambd)
        losses.append(loss.item())

        # Print progress
        print(f"Epoch {epoch + 1} - Loss: {loss.item():.4f} - MPR: {mpr_value:.4f}")

    return X, Y, beta_u, beta_i, mprs, losses

In [32]:
num_latent = 120
lambd = 0
X, Y, beta_u, beta_i, mprs, losses = train_with_gradients(
    R=R,
    num_latent=num_latent,
    num_users=num_users,
    num_items=num_items,
    lambd=lambd,
    alpha=alpha,
    learning_rate=0.1,
    epochs=500,
)

Epoch 1 - Loss: -38629.8494 - MPR: 0.3232
Epoch 2 - Loss: -17282.9525 - MPR: 0.1578
Epoch 3 - Loss: -21596.7778 - MPR: 0.1219
Epoch 4 - Loss: -9401.9801 - MPR: 0.0735
Epoch 5 - Loss: -8133.4859 - MPR: 0.0629
Epoch 6 - Loss: -4629.9198 - MPR: 0.0478
Epoch 7 - Loss: -3656.9187 - MPR: 0.0416
Epoch 8 - Loss: -2475.4362 - MPR: 0.0359
Epoch 9 - Loss: -2405.5615 - MPR: 0.0335
Epoch 10 - Loss: -1824.1566 - MPR: 0.0325
Epoch 11 - Loss: -1651.7977 - MPR: 0.0310
Epoch 12 - Loss: -1636.2626 - MPR: 0.0303
Epoch 13 - Loss: -1645.4322 - MPR: 0.0299
Epoch 14 - Loss: -1654.5976 - MPR: 0.0297
Epoch 15 - Loss: -1598.1404 - MPR: 0.0295
Epoch 16 - Loss: -1523.6853 - MPR: 0.0293
Epoch 17 - Loss: -1483.0227 - MPR: 0.0292
Epoch 18 - Loss: -1466.7401 - MPR: 0.0291
Epoch 19 - Loss: -1469.3116 - MPR: 0.0290
Epoch 20 - Loss: -1481.8341 - MPR: 0.0290
Epoch 21 - Loss: -1546.1490 - MPR: 0.0289
Epoch 22 - Loss: -1575.4245 - MPR: 0.0289
Epoch 23 - Loss: -1493.9351 - MPR: 0.0289
Epoch 24 - Loss: -1459.3900 - MPR: 0.028

In [33]:
px.line(y=mprs, title="MPR over epochs").show()
px.line(y=losses, title="Loss over epochs").show()

In [34]:
user_latent = X.detach().numpy()
item_latent = Y.detach().numpy()

In [35]:
if num_latent <= 3:
    # Add the latent vectors to the dataframe
    df_matrix_mf["user_latent"] = df_matrix_mf["username"].cat.codes.apply(
        lambda x: user_latent[x]
    )
    df_matrix_mf["item_latent"] = df_matrix_mf["id"].cat.codes.apply(
        lambda x: item_latent[x]
    )

    # Add the biases to the dataframe
    df_matrix_mf["user_bias"] = df_matrix_mf["username"].cat.codes.apply(
        lambda x: beta_u[x].item()
    )
    df_matrix_mf["item_bias"] = df_matrix_mf["id"].cat.codes.apply(
        lambda x: beta_i[x].item()
    )

    for i in range(num_latent):
        df_matrix_mf[f"user_latent_{i}"] = df_matrix_mf["user_latent"].apply(lambda x: x[i])
        df_matrix_mf[f"latent_{i}"] = df_matrix_mf["item_latent"].apply(lambda x: x[i])

    fig = plotting.plot_latent_space(
        df_matrix_mf,
        color=df_matrix_mf["username"],
        text=df_matrix_mf["username"],
        title="User latent space",
        latent_columns=["latent_0", "latent_1", "latent_2"],
    )
    fig.show()

    df_user_latent = df_matrix_mf.drop_duplicates(subset=["username"])
    fig = plotting.plot_latent_space(
        df_user_latent,
        color=df_user_latent["username"],
        text=df_user_latent["username"],
        title="User latent space",
        latent_columns=["user_latent_0", "user_latent_1", "user_latent_2"],
    )
    fig.show()

In [39]:
# Choose a user
user = "michelle"
user_id = convert_to_ids([user], "username")[0]
user_features = X[user_id]
beta_i_np = beta_i.detach().numpy()
beta_u_np = beta_u.detach().numpy()

# Recommend items to the user by computing the dot product between the user features and the item features
recommendations = np.dot(user_features, item_latent.T) + beta_u_np[user_id] + beta_i_np
df_recommendations = df_matrix_mf.copy()
df_recommendations = df_recommendations.drop_duplicates(subset=["id"])
df_recommendations["recommendation"] = recommendations
df_recommendations = df_recommendations.sort_values(by="recommendation", ascending=False)
df_recommendations = df_recommendations[df_recommendations["username"] != user]
df_recommendations[spoti.PRETTY_PRINT_FEATURES].head(50)

,username,artists_names,name,release_year,popularity,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,tempo,loudness,duration_ms,release_year,popularity
78,jaslkh,London Grammar,Wasting My Young Years,2013,58,0.546,0.3060,0.0345,0.835000,0.000748,0.0980,0.1110,126.857,-10.339,204246,2013,58
6997,brenda,Moderat,A New Error,2009,63,0.717,0.4600,0.0372,0.028700,0.785000,0.0661,0.0961,110.993,-11.326,367306,2009,63
10263,paul,Relo,PRP - Plume Reconnait Plume,2022,28,0.733,0.7530,0.3060,0.245000,0.000000,0.7040,0.4860,124.065,-7.946,308000,2022,28
6996,brenda,Discobitch,C'est beau la bourgeoisie - Radio Edit,2008,61,0.624,0.8060,0.0671,0.117000,0.000000,0.4510,0.6430,127.962,-6.404,211186,2008,61
10197,paul,Alpha Wann,LA LUMIÈRE DANS LE NOIR,2018,45,0.690,0.6750,0.3110,0.211000,0.000000,0.3060,0.6740,160.069,-7.936,168773,2018,45
84,jaslkh,The Blaze,EYES,2022,64,0.818,0.4330,0.0702,0.175000,0.004130,0.1240,0.1190,119.993,-10.853,212080,2022,64
74,jaslkh,Polo & Pan,Ani Kuni,2021,55,0.661,0.6230,0.0292,0.198000,0.602000,0.0753,0.0491,121.002,-6.157,276386,2021,55
12296,dany,The Avener,Castle In The Snow,2015,37,0.663,0.8000,0.0457,0.018200,0.000019,0.0880,0.6660,98.026,-4.843,212893,2015,37
7035,brenda,Ghost,Stay [Feat. Patrick Wilson],2023,65,0.405,0.5010,0.0283,0.174000,0.000274,0.0800,0.1420,96.574,-9.152,234220,2023,65
10235,paul,Nas,Nas Is Like,1999,69,0.634,0.8450,0.3520,0.006020,0.000005,0.0598,0.9290,94.000,-5.058,237026,1999,69


Save the model

In [ ]:
# save the model
# torch.save(X, "models/X.pt")
# torch.save(Y, "models/Y.pt")
# torch.save(beta_u, "models/beta_u.pt")
# torch.save(beta_i, "models/beta_i.pt")
# torch.save(alpha, "models/alpha.pt")
# torch.save(lambd, "models/lambd.pt")